In [2]:
# caminho + supressão de avisos 
import os, warnings
#importação de dados
import pandas as pd
import numpy as np
# conexão com oo banco
from sqlalchemy import create_engine
# aprendizado de maquina
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
# visualização de dados(dashboard)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
warnings.filterwarnings('ignore')

In [4]:
# conectar ao banco
ENGINE_URL = (
'mysql+pymysql://root:@localhost:3306/bolsa_familia'
)
engine = create_engine(ENGINE_URL, echo=False)

query =  ''' SELECT `MÊS COMPETÊNCIA`, `UF`,
             COUNT(*) AS qtd_parcelas, 
             AVG(`VALOR PARCELA`) AS valor_medio, 
             SUM(`VALOR PARCELA`) AS valor_total 
             FROM `bolsa_familia` 
             GROUP BY `MÊS COMPETÊNCIA`, `UF` '''

df = pd.read_sql(query, engine)
print(df.columns)

Index(['MÊS COMPETÊNCIA', 'UF', 'qtd_parcelas', 'valor_medio', 'valor_total'], dtype='str')


In [5]:
#EDA
df_uf = df.groupby('UF').agg(
    media_valor = ('valor_medio', 'mean'),
    total_parcelas = ('qtd_parcelas', 'sum')
).reset_index()

In [6]:
serie = df_uf['media_valor']
q1, q2, q3 = np.percentile(serie, [25, 50, 75])
iqr = q3 - q1
print(f'\nMedia: {serie.mean():.2f}')
print(f'\nMediana: {serie.median():.2f}')
print(f'\n|Q1: {q1:.2f}|  |Q3: {q3:.2f}| |IQR: {iqr:.2f}|')
print(f'\nAssimetria: {serie.skew():.3f}')
print(f'\nCurtose: {serie.kurt():.3f}')


Media: 675.38

Mediana: 664.77

|Q1: 659.32|  |Q3: 681.42| |IQR: 22.09|

Assimetria: 1.354

Curtose: 0.865


In [7]:
# aprendizado // agrupamento // não su

features = df_uf[['media_valor', 'total_parcelas']]
# boas praticas para aprendizado de maquina
scaler = StandardScaler()
features_norm = scaler.fit_transform(features)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_uf['cluster'] = kmeans.fit_predict(features_norm)
print(df_uf.groupby('cluster')[['media_valor', 'total_parcelas']])